# Loan Default Prediction

**Classification | Gaussian Naive Bayes | Imbalanced Learning | Model Evaluation**

## Project objective
Build a reproducible classification workflow to identify applicants who may default on a bank loan. The dataset contains labelled historical applicants and a separate set of applicants whose default status is unknown.

The notebook:
- separates labelled and unlabelled applicants before modelling;
- creates a stratified train/test split;
- fits preprocessing and feature selection on training folds only;
- compares baseline Gaussian Naive Bayes with random over-sampling, random under-sampling and SMOTE;
- selects a model using five-fold cross-validation on default-class F1-score;
- evaluates the selected model once on the held-out test set; and
- predicts default labels and probabilities for unlabelled applicants.

> **Important:** This is an educational portfolio project, not a credit decision system. Do not use these predictions to approve or reject real loan applications.

## 1. Setup and imports

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
TARGET = "default"

## 2. Load and inspect the data

Place the supplied CSV at `data/bankloans.csv` relative to this notebook. The fallback path supports running the notebook when the CSV is stored beside it.

In [ ]:
DATA_PATHS = [
    Path("data/bankloans.csv"),
    Path("bankloans.csv"),
]
DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find bankloans.csv. Put it in ./data/bankloans.csv "
        "or in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
print("Column types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))

print("Default label distribution (including missing labels):")
display(df[TARGET].value_counts(dropna=False).to_frame("count"))

## 3. Separate labelled and unlabelled applicants

Rows with a known `default` outcome are used for model development. Rows with a missing outcome are held aside for prediction only and are never used during training or cross-validation.

In [ ]:
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' is missing.")

labelled = df.loc[df[TARGET].notna()].copy()
unlabelled = df.loc[df[TARGET].isna()].copy()

X = labelled.drop(columns=TARGET)
y = labelled[TARGET].astype(int)
X_unlabelled = unlabelled.drop(columns=TARGET).copy()

if set(y.unique()) != {0, 1}:
    raise ValueError(f"Expected binary target labels 0 and 1; found {sorted(y.unique())}")

print(f"Labelled applicants: {len(labelled)}")
print(f"Unlabelled applicants: {len(unlabelled)}")
print("\nLabel distribution:")
display(y.value_counts().rename(index={0: "Non-default (0)", 1: "Default (1)"}).to_frame("count"))
display((y.value_counts(normalize=True) * 100).round(2).rename("percentage").to_frame())

## 4. Train-test split

The split is stratified so the proportion of defaulters is approximately preserved in both sets. No outlier limits, feature-selection decisions or sampling operations are learned from the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining default rate:", round(y_train.mean(), 3))
print("Test default rate:", round(y_test.mean(), 3))

## 5. Leakage-safe preprocessing and VIF feature selection

Two custom transformers are used inside each model pipeline:

1. `QuantileCapper` learns 5th and 95th percentile caps from the current training fold and applies those same caps to validation/test data.
2. `VIFSelector` calculates VIF using only the current training fold and iteratively removes the variable with the highest VIF until all remaining VIF values are below 10 (or only one feature remains).

Because these transformers are fitted inside the cross-validation pipeline, validation folds do not influence preprocessing or feature selection.

In [ ]:
class QuantileCapper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_quantile=0.05, upper_quantile=0.95):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = np.asarray(X_df.columns, dtype=object)
        self.lower_ = X_df.quantile(self.lower_quantile)
        self.upper_ = X_df.quantile(self.upper_quantile)
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_).copy()
        return X_df.clip(lower=self.lower_, upper=self.upper_, axis=1)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_


class VIFSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=10.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = np.asarray(X_df.columns, dtype=object)
        selected = list(X_df.columns)

        while len(selected) > 1:
            values = X_df[selected].astype(float).values
            vif_values = []
            for i in range(len(selected)):
                try:
                    vif = variance_inflation_factor(values, i)
                except Exception:
                    vif = np.inf
                vif_values.append(float(vif))

            vif_series = pd.Series(vif_values, index=selected)
            max_vif = vif_series.replace([np.inf, -np.inf], np.inf).max()

            if max_vif <= self.threshold:
                break

            # Drop the feature with the largest VIF.
            selected.remove(vif_series.idxmax())

        self.selected_features_ = selected
        self.vif_table_ = pd.DataFrame({
            "Feature": selected,
            "VIF": [
                variance_inflation_factor(X_df[selected].astype(float).values, i)
                for i in range(len(selected))
            ]
        })
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X, columns=self.feature_names_in_)
        return X_df[self.selected_features_]

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.selected_features_, dtype=object)

## 6. Compare class-imbalance strategies using five-fold cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

samplers = {
    "Baseline (no resampling)": "passthrough",
    "Random oversampling": RandomOverSampler(random_state=RANDOM_STATE),
    "Random undersampling": RandomUnderSampler(random_state=RANDOM_STATE),
    "SMOTE": SMOTE(random_state=RANDOM_STATE, k_neighbors=3),
}

scoring = {
    "accuracy": "accuracy",
    "precision_default": "precision",
    "recall_default": "recall",
    "f1_default": "f1",
}

cv_rows = []
pipelines = {}

for name, sampler in samplers.items():
    pipe = Pipeline(steps=[
        ("outlier_capping", QuantileCapper()),
        ("vif_selection", VIFSelector(threshold=10.0)),
        ("sampler", sampler),
        ("model", GaussianNB()),
    ])
    pipelines[name] = pipe

    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        error_score="raise"
    )

    cv_rows.append({
        "Strategy": name,
        "Mean Accuracy": scores["test_accuracy"].mean(),
        "Mean Default Precision": scores["test_precision_default"].mean(),
        "Mean Default Recall": scores["test_recall_default"].mean(),
        "Mean Default F1": scores["test_f1_default"].mean(),
        "Std Default F1": scores["test_f1_default"].std(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values(
    "Mean Default F1", ascending=False
).reset_index(drop=True)

display(cv_results.round(3))

### Model selection

The strategy with the highest mean default-class F1-score across the five validation folds is selected. F1 balances precision and recall; for lending-risk screening, recall is also important because missed defaulters can be costly. The appropriate trade-off depends on the real business context.

In [ ]:
best_strategy = cv_results.loc[0, "Strategy"]
best_pipeline = pipelines[best_strategy]

print("Selected strategy:", best_strategy)

## 7. Fit selected model and evaluate on the untouched test set

In [ ]:
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Default precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Default recall: {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Default F1-score: {f1_score(y_test, y_pred, zero_division=0):.3f}")

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    labels=[0, 1],
    target_names=["Non-default (0)", "Default (1)"],
    zero_division=0
))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-default", "Default"]
)
disp.plot(values_format="d")
plt.title(f"Test Confusion Matrix — {best_strategy}")
plt.show()

## 8. Inspect selected features and fitted outlier limits

These values are learned from the complete training partition after the best strategy is selected. The cross-validation results above remain the basis for comparing sampling strategies.

In [ ]:
capper = best_pipeline.named_steps["outlier_capping"]
vif_selector = best_pipeline.named_steps["vif_selection"]

print("Features retained by VIF selection:")
print(vif_selector.selected_features_)
display(vif_selector.vif_table_.round(3))

print("Outlier capping limits fitted on training data:")
display(pd.DataFrame({"lower_5pct": capper.lower_, "upper_95pct": capper.upper_}).round(3))

## 9. Predict applicants with unknown default status

The fitted pipeline applies the same training-derived outlier limits and selected features to the unlabelled applicants. It then returns both a predicted class and the estimated probability of default.

In [ ]:
if len(X_unlabelled) > 0:
    unlabelled_predictions = best_pipeline.predict(X_unlabelled)
    unlabelled_probabilities = best_pipeline.predict_proba(X_unlabelled)[:, 1]

    predictions = unlabelled.copy()
    predictions["predicted_default"] = unlabelled_predictions
    predictions["default_probability"] = unlabelled_probabilities

    display(predictions.head())

    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / "loan_default_predictions.csv"
    predictions.to_csv(output_path, index=False)
    print(f"Saved {len(predictions)} predictions to: {output_path}")
else:
    print("No unlabelled applicants were found; no prediction file was created.")

## 10. Conclusions and limitations

- The project compares four approaches to class imbalance rather than relying on accuracy alone.
- Model selection is based on five-fold cross-validation of default-class F1-score.
- The held-out test set is reserved for final evaluation.
- Preprocessing, VIF feature selection and resampling are fitted within each training fold to reduce leakage.
- Gaussian Naive Bayes is a simple baseline; additional models such as Logistic Regression, Random Forest and gradient boosting could be evaluated in future work.
- The dataset may not represent current lending populations. Results should not be interpreted as evidence of fairness, calibration or suitability for real credit decisions.

**Reproducibility:** The random seed is fixed at 42. Install dependencies from `requirements.txt` before running the notebook.